In [1]:
# Aggregate dataset to Plant,Material,Channel level
# Reuse the clean, processed file created for the lowest granularity - Plant,Material,Channel,DPGrp

import os
import sys
import time
import pandas as pd
import numpy as np

pd.set_option("display.max_colwidth", 2000)

In [2]:
cleaned_input_filepath = "../data/plant_mat_channel_agg_transitions_input.xlsx"
aggregated_output_filepath = "../data/plant_mattransgrp_channel_agg_input.xlsx"

##### Aggregate keys from Plant-Material-DC-Customer to Plant-Material-DC level

In [3]:
df = pd.read_excel(cleaned_input_filepath)

In [4]:
df.head()

,Plant,Material,Channel,YearMonth,Customer Demand Qty,Customer Demand Value,Working_Days,Material_Desc,Material_Group,Material_Group_Desc,Material_Hierarchy,Material_Hierarchy_Desc,Valuation_Class,Valuation_Class_Desc,Minimum Lot Size,key,transition_materials_group
0,3891,0204802915,10,2022-01-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"('0204802915',)"
1,3891,0204802915,10,2022-02-01,0,0.0,16,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"('0204802915',)"
2,3891,0204802915,10,2022-03-01,0,0.0,23,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"('0204802915',)"
3,3891,0204802915,10,2022-04-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"('0204802915',)"
4,3891,0204802915,10,2022-05-01,0,0.0,19,Gear shift Unit,PG016,Transmission,16ED,Add-on Gear Actuatio,3100,Trading goods,80,3891_0204802915_10,"('0204802915',)"


In [5]:
df.dtypes

Plant                                  int64
Material                                 str
Channel                                int64
YearMonth                     datetime64[us]
Customer Demand Qty                    int64
Customer Demand Value                float64
Working_Days                           int64
Material_Desc                            str
Material_Group                           str
Material_Group_Desc                      str
Material_Hierarchy                       str
Material_Hierarchy_Desc                  str
Valuation_Class                        int64
Valuation_Class_Desc                     str
Minimum Lot Size                       int64
key                                      str
transition_materials_group               str
dtype: object

In [6]:
# Verify uniqueness of attributes at key level
KEY = ["Plant", "transition_materials_group", "Channel"]
DROP_COLS = ['Material', 'Material_Desc', 'Valuation_Class', 'Valuation_Class_Desc', 'Minimum Lot Size', "DP_Group_L3", "DP_Group_L3_Desc", "key"]
MEASURES = ["Customer Demand Qty", "Customer Demand Value"]
TIME = ["YearMonth"]

# Working_Days: treat as "time-varying attribute" to be averaged per KEY+TIME
TIME_VARYING_ATTRS = ["Working_Days"]  # extendable list if needed later

df_chk = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).copy()

attr_cols = [c for c in df_chk.columns if c not in KEY + TIME + MEASURES]

violations = {}
for c in attr_cols:
    nun = df_chk.groupby(KEY)[c].nunique(dropna=True)
    bad = nun[nun > 1]
    if not bad.empty:
        violations[c] = bad

len(violations), list(violations.keys())

(1, ['Working_Days'])

In [7]:


def aggregate_fast_validated(df: pd.DataFrame) -> pd.DataFrame:
    # 1) Drop unwanted columns
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).copy()

    # 2) Ensure datetime
    df["YearMonth"] = pd.to_datetime(df["YearMonth"])

    group_cols = KEY + TIME

    # 3) Identify candidate attribute columns (everything except keys/time/measures)
    attr_cols = [c for c in df.columns if c not in group_cols + MEASURES]

    # Split attributes into:
    # - static per KEY (validate uniqueness per KEY)
    # - time-varying per KEY+TIME (aggregate per KEY+TIME)
    time_varying = [c for c in TIME_VARYING_ATTRS if c in df.columns]
    static_attrs = [c for c in attr_cols if c not in time_varying]

    # A) FAST UNIQUENESS VALIDATION for STATIC attributes at KEY level
    if static_attrs:
        nun = df.groupby(KEY, dropna=False)[static_attrs].nunique(dropna=True)
        violating_cols = nun.columns[(nun > 1).any()].tolist()

        if violating_cols:
            diag = {}
            for c in violating_cols[:10]:
                bad_keys = nun.index[nun[c] > 1].tolist()[:5]
                diag[c] = bad_keys

            raise ValueError(
                f"Static attribute(s) not unique per key {KEY}. "
                f"Violating columns: {violating_cols}\n"
                f"Sample offending keys (first 5): {diag}\n"
                f"Note: '{time_varying}' are excluded from static validation."
            )

        # Extract static attributes once per KEY (safe because validated)
        df_static = df.groupby(KEY, as_index=False, dropna=False)[static_attrs].first()
    else:
        df_static = df[KEY].drop_duplicates()

    # B) Aggregate measures + time-varying attributes at KEY+TIME level
    agg_dict = {m: "sum" for m in MEASURES}
    # Working_Days mean per month per key
    for c in time_varying:
        agg_dict[c] = "mean"

    df_km = (
        df.groupby(group_cols, as_index=False, dropna=False)
          .agg(agg_dict)
    )

    # C) Join static attrs back onto monthly fact table
    out = df_km.merge(df_static, on=KEY, how="left")

    # Optional: cast Working_Days to int
    if "Working_Days" in out.columns:
        # If mean resulted in 22.0 etc., safe to round and convert
        # Comment out if you want float
        out["Working_Days"] = out["Working_Days"].round().astype("Int64")

    # Sanity: enforce unique grain
    dup = out.duplicated(group_cols).sum()
    if dup:
        raise AssertionError(f"Unexpected duplicates after aggregation at {group_cols}: {dup}")

    return out

df_agg = aggregate_fast_validated(df)


In [8]:
df_agg.shape

(375576, 11)

In [9]:
df_agg.head()

,Plant,transition_materials_group,Channel,YearMonth,Customer Demand Qty,Customer Demand Value,Working_Days,Material_Group,Material_Group_Desc,Material_Hierarchy,Material_Hierarchy_Desc
0,3891,"('0204802915',)",10,2022-01-01,0,0.0,19,PG016,Transmission,16ED,Add-on Gear Actuatio
1,3891,"('0204802915',)",10,2022-02-01,0,0.0,16,PG016,Transmission,16ED,Add-on Gear Actuatio
2,3891,"('0204802915',)",10,2022-03-01,0,0.0,23,PG016,Transmission,16ED,Add-on Gear Actuatio
3,3891,"('0204802915',)",10,2022-04-01,0,0.0,19,PG016,Transmission,16ED,Add-on Gear Actuatio
4,3891,"('0204802915',)",10,2022-05-01,0,0.0,19,PG016,Transmission,16ED,Add-on Gear Actuatio


In [10]:
# Verify Each (Plant, Material, Channel, YearMonth) should appear once

assert df_agg.duplicated(["Plant","transition_materials_group","Channel","YearMonth"]).sum() == 0

In [11]:
# create key col
df_agg['key'] = df_agg['Plant'].astype(str) + \
'_' + df_agg['transition_materials_group'].astype(str) + \
'_' + df_agg['Channel'].astype(str) 

# check nulls (shouldn't be present in categorical or datetime columns)
print(df_agg.isnull().any())

# write out
df_agg.to_excel(aggregated_output_filepath, index=False)

Plant                         False
transition_materials_group    False
Channel                       False
YearMonth                     False
Customer Demand Qty           False
Customer Demand Value         False
Working_Days                  False
Material_Group                False
Material_Group_Desc           False
Material_Hierarchy            False
Material_Hierarchy_Desc       False
key                           False
dtype: bool
